In [46]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn import preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


/kaggle/input/datasets/kapturovalexander/customers-churned-in-telecom-services/customer_churn_telecom_services.csv


In [2]:
path = "/kaggle/input/datasets/kapturovalexander/customers-churned-in-telecom-services/customer_churn_telecom_services.csv"
df = pd.read_csv(path)
print(df)

      gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0     Female              0     Yes         No       1           No   
1       Male              0      No         No      34          Yes   
2       Male              0      No         No       2          Yes   
3       Male              0      No         No      45           No   
4     Female              0      No         No       2          Yes   
...      ...            ...     ...        ...     ...          ...   
7038    Male              0     Yes        Yes      24          Yes   
7039  Female              0     Yes        Yes      72          Yes   
7040  Female              0     Yes        Yes      11           No   
7041    Male              1     Yes         No       4          Yes   
7042    Male              0      No         No      66          Yes   

         MultipleLines InternetService OnlineSecurity OnlineBackup  \
0     No phone service             DSL             No          Yes   
1      

# Dados faltantes
Temos 11 valores faltantes na coluna TotalCharges. Essa coluna corresponde ao total cobrado de cada cliente. Abaixo, verificamos que esses 11 casos correspondem aos 11 casos nos quais tenure (número de meses que o cliente passou com a empresa) é 0. Podemos assumir que nesses casos TotalCharges deveria ser 0, pois ainda não foi cobrado o valor mensal. 

In [3]:
dfna = df.isnull()

print(dfna.sum())

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [4]:
linhas_faltantes = df[df["TotalCharges"].isnull()]
#print(linhas_faltantes)

tenure_zero = df[df["tenure"] == 0]
#print(df2)

tenure_zero.compare(linhas_faltantes)

Empty DataFrame
Columns: []
Index: []

In [5]:
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

# Análise de inconsistências
Analisamos se existe algum caso no qual o valor da coluna PhoneService e InternetService é contradito na coluna de algum dos serviços opcionais. Não foram encontradas inconsistências.

In [6]:
condicao_1 = (df['PhoneService'] == 'No') & (df['MultipleLines'] != 'No phone service')
condicao_2 = (df['PhoneService'] == 'Yes') & (df['MultipleLines'] == 'No phone service')

inconsistencias = df[condicao_1 | condicao_2]
print(f"Número de casos inconsistentes: {len(inconsistencias)}")

Número de casos inconsistentes: 0


In [7]:
cols_teste = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]
for col in cols_teste:
    condicao_1 = (df['InternetService'] == "No") & (df[col] != "No internet service")
    condicao_2 = (df['InternetService'] != "No") & (df[col] == "No internet service")

    inconsistencias = df[condicao_1 | condicao_2]
    print(f"Número de casos inconsistentes {col}: {len(inconsistencias)}")

Número de casos inconsistentes OnlineSecurity: 0
Número de casos inconsistentes OnlineBackup: 0
Número de casos inconsistentes DeviceProtection: 0
Número de casos inconsistentes TechSupport: 0
Número de casos inconsistentes StreamingTV: 0
Número de casos inconsistentes StreamingMovies: 0


# Codificação de variáveis categóricas

Variáveis categóricas e forma de codificação:
- gender -> binária
- Partner -> binária
- Dependents -> binária
- PhoneService -> binária
- MultipleLines -> binária (dropa a coluna no service)
- InternetServices -> one-hot
- OnlineSecurity -> binário (dropa a coluna no service)
- OnlineBackup -> binário (dropa a coluna no service)
- DeviceProtection -> binário (dropa a coluna no service)
- TechSupport -> binário (dropa a coluna no service)
- StreamingTV -> binário (dropa a coluna no service)
- StreamingMovies -> binário (dropa a coluna no service)
- Contract -> one-hot
- PaperlessBilling -> binária
- PaymentMethod -> one-hot
- Churn -> binária

podemos dropar a coluna no service do one-hot encoding pois a informação contida nela está replicada na variável InternetServices ou PhoneService. Isso geraria apenas colunas yes/no, que podem ser tratadas com codificação binária, reduzindo a dimensionalidade quando comparado com fazer one-hot em todas essas categorias. Essa decisão é respaldada pela análise de inconsistências.

In [8]:
cols = df.columns

for col in cols:
    unique = df[col].unique()
    print(f"{col}: {unique} ({len(unique)})")

gender: ['Female' 'Male'] (2)
SeniorCitizen: [0 1] (2)
Partner: ['Yes' 'No'] (2)
Dependents: ['No' 'Yes'] (2)
tenure: [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39] (73)
PhoneService: ['No' 'Yes'] (2)
MultipleLines: ['No phone service' 'No' 'Yes'] (3)
InternetService: ['DSL' 'Fiber optic' 'No'] (3)
OnlineSecurity: ['No' 'Yes' 'No internet service'] (3)
OnlineBackup: ['Yes' 'No' 'No internet service'] (3)
DeviceProtection: ['No' 'Yes' 'No internet service'] (3)
TechSupport: ['No' 'Yes' 'No internet service'] (3)
StreamingTV: ['No' 'Yes' 'No internet service'] (3)
StreamingMovies: ['No' 'Yes' 'No internet service'] (3)
Contract: ['Month-to-month' 'One year' 'Two year'] (3)
PaperlessBilling: ['Yes' 'No'] (2)
PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)'] (4

In [9]:
binaria_cols = {
    'gender': {'Female': 0, 'Male': 1},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1},
    'Churn': {'No': 0, 'Yes': 1}
}

for col, mapping in binaria_cols.items():
    df[col] = df[col].map(mapping)

df['MultipleLines'] = df['MultipleLines'].map({'Yes': 1, 'No': 0, 'No phone service': 0})

addons_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

for col in addons_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No internet service': 0})

nominal_cols = ['InternetService', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=False, dtype=int)

In [10]:
print(f"Dimensões finais do DataFrame após o encoding: {df.shape}")
display(df.head())
print(df.columns)

Dimensões finais do DataFrame após o encoding: (7043, 27)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,...,1,0,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,1,0,1,...,1,0,0,0,1,0,0,0,0,1
2,1,0,0,0,2,1,0,1,1,0,...,1,0,0,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,1,0,1,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,0,2,1,0,0,0,0,...,0,1,0,1,0,0,0,0,1,0


Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'Contract_Month-to-month', 'Contract_One year',
       'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='object')


# Outliers

Não foram detectados outliers pelo critério IQR, e a análise dos valores únicos de cada coluna descarta a possibilidade de outliers nos valores categóricos.

In [11]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

display(df[num_cols].describe())

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < limite_inferior) | (df[col] > limite_superior)]
    print(f"Coluna {col}: {len(outliers)} outliers encontrados pelo critério IQR.")

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000
mean,32.371149,64.761692,2279.734304
std,24.559481,30.090047,2266.794470
min,0.000000,18.250000,0.000000
25%,9.000000,35.500000,398.550000
50%,29.000000,70.350000,1394.550000
75%,55.000000,89.850000,3786.600000
max,72.000000,118.750000,8684.800000


Coluna tenure: 0 outliers encontrados pelo critério IQR.
Coluna MonthlyCharges: 0 outliers encontrados pelo critério IQR.
Coluna TotalCharges: 0 outliers encontrados pelo critério IQR.


# Variáveis agregadas/transformadas

Candidatos: 
- Número de addons assinados:
   total_addons = MultipleLines + OnlineSecurity + OnlineBackup + DeviceProtection + TechSupport + StreamingTV + StreamingMovies
- Pagamento automatico:
    Payment_Method_Automatic = 'PaymentMethod_Bank transfer (automatic)' || 'PaymentMethod_Credit card (automatic)'

In [16]:
addons_cols = [
    'MultipleLines','OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['total_addons'] = df[addons_cols].sum(axis=1)


auto_payment_cols = [col for col in df.columns if 'PaymentMethod_' in col and 'automatic' in col.lower()]
df['PaymentMethod_Automatic'] = df[auto_payment_cols].max(axis=1)

print(f"Dimensões finais do DataFrame após o encoding e criação de features: {df.shape}")
display(df[['total_addons', 'PaymentMethod_Automatic', 'TotalCharges']].head())

Dimensões finais do DataFrame após o encoding e criação de features: (7043, 29)


,total_addons,PaymentMethod_Automatic,TotalCharges
0,1,0,29.85
1,2,0,1889.50
2,2,0,108.15
3,3,1,1840.75
4,0,0,151.65


# Divisão dos conjuntos de treino e teste

O teste é separado uma única vez, estratificado por Churn, e só é usado no fim. O treino fica como está: um cliente por linha, sem balanceamento e sem padronização. A validação vem do k-fold feito sobre esse treino original, na seção seguinte.

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

print("\nProporção de Churn no treino (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nProporção de Churn no teste (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

# Normalização e balanceamento dentro do k-fold

O k-fold é feito no treino original, antes do StandardScaler e do RandomOverSampler. Em cada dobra, o scaler é ajustado só na parte de treino e o oversampling só duplica essa parte. A validação da dobra continua com um cliente por linha e só recebe o `transform`.

Fazer o k-fold em um treino já balanceado e padronizado espalharia as cópias de um mesmo cliente entre o treino e a validação da dobra, e a métrica subiria sem refletir clientes novos.

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

def preparar(X_tr, y_tr, X_aval):
    # Scaler ajustado só em X_tr; X_aval só recebe o transform e não é balanceado
    scaler = StandardScaler()

    X_tr_scaled = X_tr.copy()
    X_tr_scaled[numeric_cols] = scaler.fit_transform(X_tr[numeric_cols])

    X_aval_scaled = X_aval.copy()
    X_aval_scaled[numeric_cols] = scaler.transform(X_aval[numeric_cols])

    ros = RandomOverSampler(random_state=42)
    X_tr_balanced, y_tr_balanced = ros.fit_resample(X_tr_scaled, y_tr)

    return X_tr_balanced, y_tr_balanced, X_aval_scaled

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []
for i, (idx_tr, idx_val) in enumerate(skf.split(X_train, y_train)):
    X_fold_val, y_fold_val = X_train.iloc[idx_val], y_train.iloc[idx_val]
    X_fold_train, y_fold_train, X_fold_val = preparar(
        X_train.iloc[idx_tr], y_train.iloc[idx_tr], X_fold_val
    )
    folds.append((X_fold_train, y_fold_train, X_fold_val, y_fold_val))

    print(f"Dobra {i}: treino {X_fold_train.shape} "
          f"(churn {y_fold_train.mean() * 100:.1f}%), "
          f"validação {X_fold_val.shape} (churn {y_fold_val.mean() * 100:.1f}%)")